# Dupire Local Volatility

## The problem local vol solves

BSM assumes a single constant $\sigma$. But the market smile shows implied vol varies
by strike and maturity - the market's terminal distribution of $S_T$ is *not* lognormal
(skewed, fat left tail: equity crash fear). A constant-$\sigma$ model produces a *flat*
smile and cannot fit this.

**Local volatility** replaces the constant $\sigma$ with a deterministic *function*
$\sigma_{\text{loc}}(S, t)$ - volatility that depends on spot and time, but adds no new
source of randomness. The risk-neutral SDE becomes:

$$dS_t = (r - q)\,S_t\,dt + \sigma_{\text{loc}}(S_t, t)\,S_t\,dW_t$$

**Dupire's theorem**: given the full surface of observed call prices $C(K, T)$, there is
a *unique* local-vol function reproducing every price simultaneously, recoverable by
differentiating the surface.

## What this is, and is NOT

- **Still** one-factor (a single $W_t$), arbitrage-free, and complete - so the entire
  risk-neutral pricing framework carries over.
- **No longer** geometric Brownian motion: with $\sigma_{\text{loc}}(S,t)$ varying, the
  log-SDE has non-constant coefficients, so $S_T$ is **not lognormal** and there is **no
  closed-form price** in general.
- BSM is the special case $\sigma_{\text{loc}} = \text{const}$. Local vol is the strictly
  larger model that contains it - and abandoning the constant-$\sigma$/lognormal
  straitjacket is *precisely* what lets it fit the smile.

The non-lognormality is the point, not a flaw: it is the only way to match a market whose
implied $S_T$ distribution is non-lognormal.

## Derivation roadmap

Three pieces combine to give Dupire's formula:
1. **Fokker-Planck** - the PDE for how the risk-neutral density evolves forward in time.
2. **Breeden-Litzenberger** - links that (unobservable) density to (observable) call prices.
3. **Substitute and solve** for $\sigma_{\text{loc}}^2$.

## 1. Fokker-Planck (forward Kolmogorov equation)

A diffusion process can be described by two dual PDEs for its transition density
$p(x_0, t_0; x, t)$ = the probability of being at $x$ at time $t$ given a start at
$(x_0, t_0)$:

- **Backward Kolmogorov** differentiates the *initial* variables $(x_0, t_0)$: fix where
  you observe, ask how the density depends on where you started. Pricing PDEs (like BSM)
  are backward equations - they propagate a fixed terminal payoff *back* to today.
- **Forward Kolmogorov = Fokker-Planck** differentiates the *terminal* variables $(x, t)$:
  fix the (known) starting point, watch the density evolve *forward* in time.## 2. Breeden-Litzenberger: density from prices

The density $p$ in Fokker-Planck is unobservable. This result expresses it in terms of
observable call prices. Start from the discounted-expected-payoff definition:

$$C(K, T) = e^{-rT}\int_K^\infty (S - K)\,p(S_0, t_0; S, T)\,dS$$

Differentiate once in $K$ (Leibniz rule - the $(S-K)$ term and the lower limit both
depend on $K$; the boundary term vanishes since the integrand is zero at $S=K$):

$$\frac{\partial C}{\partial K} = -e^{-rT}\int_K^\infty p\,dS$$

Differentiate again:

$$\boxed{\;\frac{\partial^2 C}{\partial K^2} = e^{-rT}\,p(S_0, t_0; K, T)\;}$$

**The second strike-derivative of the call price IS the discounted risk-neutral density**,
evaluated at $S = K$. This is the bridge: it lets us replace $p$ in Fokker-Planck with
$\partial^2 C / \partial K^2$, turning a statement about densities into a formula in
prices we can measure.

For a diffusion $dX = \mu(X,t)\,dt + \sigma(X,t)\,dW$, Fokker-Planck reads:

$$\frac{\partial p}{\partial t} = -\frac{\partial}{\partial x}\big[\mu(x,t)\,p\big] + \frac12\frac{\partial^2}{\partial x^2}\big[\sigma^2(x,t)\,p\big]$$

Two terms with a clear physical reading:
- **Drift / advection** $-\partial_x[\mu p]$: the density's centre of mass moving.
- **Diffusion** $\tfrac12\partial_{xx}[\sigma^2 p]$: the density *spreading*, governed by
  $\sigma^2$. **This is the only term carrying $\sigma^2$** - the quantity we want to extract.

**Why Dupire needs the forward equation specifically**: it is parameterised by the
*terminal* variables. For options, the terminal variables are the **strike $K$ and
maturity $T$** - exactly the axes of the price surface we observe. The forward equation
lets us sweep the whole $(K, T)$ surface from a single starting point (today's spot),
which the backward equation (good for one option at a time) cannot.

## 2. Breeden-Litzenberger: density from prices

The density $p$ in Fokker-Planck is unobservable. This result expresses it in terms of
observable call prices. Start from the discounted-expected-payoff definition:

$$C(K, T) = e^{-rT}\int_K^\infty (S - K)\,p(S_0, t_0; S, T)\,dS$$

Differentiate once in $K$ (Leibniz rule - the $(S-K)$ term and the lower limit both
depend on $K$; the boundary term vanishes since the integrand is zero at $S=K$):

$$\frac{\partial C}{\partial K} = -e^{-rT}\int_K^\infty p\,dS$$

Differentiate again:

$$\boxed{\;\frac{\partial^2 C}{\partial K^2} = e^{-rT}\,p(S_0, t_0; K, T)\;}$$

**The second strike-derivative of the call price IS the discounted risk-neutral density**,
evaluated at $S = K$. This is the bridge: it lets us replace $p$ in Fokker-Planck with
$\partial^2 C / \partial K^2$, turning a statement about densities into a formula in
prices we can measure.

## 3. Dupire's formula

Substituting Breeden-Litzenberger into the forward equation for $C(K,T)$ and solving for
the diffusion coefficient gives:

$$\sigma_{\text{loc}}^2(K, T) = \frac{\dfrac{\partial C}{\partial T} + (r - q)\,K\,\dfrac{\partial C}{\partial K} + q\,C}{\tfrac12\,K^2\,\dfrac{\partial^2 C}{\partial K^2}}$$

**How to read it structurally** (so it is understood, not memorised):

- **Denominator** $\tfrac12 K^2 \partial^2 C/\partial K^2$ is *literally the diffusion term
  of Fokker-Planck*, with the density rewritten as $\partial^2 C/\partial K^2$. Since
  $\sigma^2$ *multiplied* the density term in the PDE, isolating it means *dividing* by
  that term, hence the density lands in the denominator.
- **Numerator** collects the time-evolution and drift terms: $\partial C/\partial T$ is
  the forward time-derivative ($\partial p/\partial t$), and $(r-q)K\,\partial C/\partial K + qC$
  are the drift pieces.

## The practical curse (why this is hard, not just elegant)

The density sits in the **denominator**, and $\partial^2 C/\partial K^2$ *is* that density
(Breeden-Litzenberger). In the far wings (deep OTM), the density is tiny - the stock is
very unlikely to finish there - so $\partial^2 C/\partial K^2 \to 0$. Dividing by a
near-zero number means:

> A small error in the second derivative of a *noisy* price surface, divided by a
> near-zero denominator, produces a wildly unstable $\sigma_{\text{loc}}$.

The formula is **exact in theory, a minefield in practice**. This is why local-vol
extraction demands a **smooth, arbitrage-free price/IV surface** *before* differentiating -
raw market quotes are far too noisy. Building that surface (interpolation, arbitrage
constraints, SVI or similar) is the real engineering work of local vol, and the reason
the denominator structure matters: it tells you exactly where the model will blow up.

In [1]:
from models.bsm import bsm_price
from calibration.dupire import dupire_local_vol
import numpy as np

S0, r, q, sigma_true = 100.0, 0.05, 0.0, 0.20
K_grid = np.linspace(80, 120, 41)      # uniform strikes
T_grid = np.linspace(0.1, 2.0, 40)     # uniform maturities

# build C(K,T) grid from closed-form BSM (constant vol)
C = np.array([[bsm_price(S0, K, T, r, sigma_true, 'call') for K in K_grid]
              for T in T_grid])         # shape (n_T, n_K)

sigma_loc = dupire_local_vol(C, K_grid, T_grid, r, q)

print(f"mean {np.nanmean(sigma_loc):.4f}, std {np.nanstd(sigma_loc):.4f}, "
      f"min {np.nanmin(sigma_loc):.4f}, max {np.nanmax(sigma_loc):.4f}")

Dupire extraction: 1640 grid points
  masked (absolute density < 1e-06): 0
  masked (relative, < 0.01 x row peak): 3
  valid points: 1637 (99.8%)
mean 0.2001, std 0.0055, min 0.1668, max 0.2802


In [2]:
# where are the bad points? print the surface's deviation from 0.20
deviation = np.abs(sigma_loc - 0.20)
bad_T, bad_K = np.unravel_index(np.argmax(deviation), deviation.shape)
print(f"worst point: T={T_grid[bad_T]:.3f}, K={K_grid[bad_K]:.1f}, "
      f"sigma_loc={sigma_loc[bad_T, bad_K]:.4f}")

# and check: does trimming the wings clean it up?
interior = sigma_loc[1:-1, 5:-5]   # drop edge maturities and far strikes
print(f"interior only: mean {interior.mean():.4f}, std {interior.std():.4f}, "
      f"min {interior.min():.4f}, max {interior.max():.4f}")

worst point: T=0.100, K=80.0, sigma_loc=nan
interior only: mean 0.2000, std 0.0004, min 0.1970, max 0.2016


## Checkpoint 1: PDE pricer validation against Black-Scholes

Before trusting the local-vol PDE pricer on a *varying* surface, validate it on the
one case with a closed-form answer: **constant volatility**, where the PDE must
reproduce `bsm_price`. We check two things:

1. **Accuracy** - the price matches BSM at a reference grid.
2. **Convergence** - Crank-Nicolson is $O(\Delta x^2, \Delta t^2)$, so the error should
   shrink ~4x each time the grid is doubled (second-order). A plateau instead of
   quartering would signal a first-order leak (often a boundary bug).

Only once the pricer reproduces BSM *and* converges at the right rate do we trust it to
generate prices under a non-trivial $\sigma_{\text{loc}}(S,t)$ for the Dupire round-trip.

In [3]:
from pricing.pde import price_localvol

# reference case
S0, K, T, r, q, sigma = 100.0, 100.0, 1.0, 0.05, 0.0, 0.20
const_vol = lambda S, t: sigma * np.ones_like(S)
bsm = bsm_price(S0, K, T, r, sigma, 'call')

# --- accuracy at the reference grid ---
pde = price_localvol(K=K, S0=S0, T=T, r=r, q=q,
                          local_vol_fn=const_vol, sigma_max=sigma, n_S=200, n_t=200)
print(f"Reference (200x200): PDE {pde:.4f} vs BSM {bsm:.4f}  diff {abs(pde-bsm):.2e}\n")

put_pde = price_localvol(K=K, S0=S0, T=T, r=r, q=q,
                         local_vol_fn=const_vol, sigma_max=sigma, n_S=200, n_t=200,
                         option_type='put')
put_bsm = bsm_price(100, 100, 1.0, 0.05, 0.20, 'put')
print(f"PUT: PDE {put_pde:.4f} vs BSM {put_bsm:.4f}  diff {abs(put_pde-put_bsm):.2e}")

# --- convergence: error should quarter each time n doubles ---
print(f"{'n':>5} {'PDE price':>12} {'abs error':>12} {'ratio':>8}")
prev_err = None
for n in [50, 100, 200, 400]:
    p = price_localvol(K=K, S0=S0, T=T, r=r, q=q,
                            local_vol_fn=const_vol, sigma_max=sigma, n_S=n, n_t=n)
    err = abs(p - bsm)
    ratio = prev_err / err if prev_err else np.nan
    print(f"{n:>5} {p:>12.5f} {err:>12.2e} {ratio:>8.2f}")
    prev_err = err

Reference (200x200): PDE 10.4530 vs BSM 10.4506  diff 2.40e-03

PUT: PDE 5.5759 vs BSM 5.5735  diff 2.40e-03
    n    PDE price    abs error    ratio
   50     10.49133     4.07e-02      nan
  100     10.46051     9.93e-03     4.11
  200     10.45298     2.40e-03     4.13
  400     10.45110     5.18e-04     4.63


In [4]:
interior = sigma_loc[2:-2, 2:-2]
sd = interior[:, 2:] - 2*interior[:, 1:-1] + interior[:, :-2]
print(f"clean second-diff max: {np.nanmax(np.abs(sd)):.2e}, threshold: 1e-3")

clean second-diff max: 1.66e-04, threshold: 1e-3


## Checkpoint 2: PDE pricer validation against Linear Skew Volatility function


In [4]:
def linear_skew(S, t, S0=100.0, sigma0=0.2, beta=-0.5):
    sig = sigma0 * (1 + beta * np.log(S / S0))
    return np.clip(sig, 0.05, 0.60)        # keep vol physical in the wings

S0, r, q = 100.0, 0.05, 0.0
sigma_max = 0.25                      # the clip ceiling — sets PDE grid width

# # --- price a C(K,T) grid under linear_skew via the PDE ---
# K_grid = np.linspace(85, 115, 31)     # interior strikes (avoid extreme wings)
# T_grid = np.linspace(0.2, 1.5, 27)    # maturities (avoid very short T)

# C = np.zeros((len(T_grid), len(K_grid)))
# for j, K in enumerate(K_grid):
#     for i, T in enumerate(T_grid):
#         C[i, j] = price_call_localvol(K=K, S0=S0, T=T, r=r, q=q,
#                                       local_vol_fn=linear_skew,
#                                       sigma_max=sigma_max, n_S=200, n_t=200)

# # --- extract local vol from those prices ---
# sigma_recovered = dupire_local_vol(C, K_grid, T_grid, r, q, verbose=True)

# # --- compare to ground truth: linear_skew(K, T) at each node ---
# # Dupire gives sigma_loc(K,T); truth is the surface at spot=K
# KK = K_grid[np.newaxis, :]            # broadcast over the (T,K) grid
# sigma_true = linear_skew(KK, None)    # time-independent here, so t unused

# # interior comparison (trim edges where FD + denominator are weakest)
# err = np.abs(sigma_recovered - sigma_true)
# interior_err = err[2:-2, 2:-2]
# print(f"\nInterior recovery: max err {np.nanmax(interior_err):.4f}, "
#       f"mean err {np.nanmean(interior_err):.4f}")
# print(f"True surface ranges {sigma_true.min():.3f} to {sigma_true.max():.3f}")    

In [5]:
# err = np.abs(sigma_recovered - sigma_true)
# err_interior = err[2:-2, 2:-2]
# i, j = np.unravel_index(np.nanargmax(err_interior), err_interior.shape)
# # map back to full grid indices (we trimmed 2 from each edge)
# print(f"worst interior point: T={T_grid[i+2]:.3f}, K={K_grid[j+2]:.1f}, "
#       f"recovered={sigma_recovered[i+2, j+2]:.4f}, true={sigma_true[0, j+2]:.4f}")

In [6]:
# # re-extract but also get the density to filter the comparison
# # (quick: recompute d2C/dK2 the same way dupire does, to build a health mask)
# dCdK = np.gradient(C, K_grid, axis=1)
# d2CdK2 = np.gradient(dCdK, K_grid, axis=1)

# # "healthy" = density comfortably above noise, not just above the hard floor
# density_health = d2CdK2 > 1e-4          # stricter than the 1e-6 extraction floor

# err = np.abs(sigma_recovered - sigma_true)
# healthy_err = err[density_health & ~np.isnan(err)]

# print(f"healthy points: {density_health.sum()} / {err.size}")
# print(f"recovery on healthy region: max {healthy_err.max():.4f}, "
#       f"mean {healthy_err.mean():.4f}")
# print(len(K_grid) == 31)

In [7]:
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# # pick a mid maturity slice to see the K-dependence clearly
# i_mid = len(T_grid) // 2
# axes[0].plot(K_grid, sigma_recovered[i_mid, :], 'o-', label='recovered')
# axes[0].plot(K_grid, sigma_true[0, :], 's-', label='true')
# axes[0].set_title(f'vol vs strike at T={T_grid[i_mid]:.2f}')
# axes[0].set_xlabel('K'); axes[0].legend(); axes[0].grid(alpha=0.3)

# # full recovered surface as a heatmap
# im1 = axes[1].imshow(sigma_recovered, aspect='auto', origin='lower',
#                      extent=[K_grid[0], K_grid[-1], T_grid[0], T_grid[-1]])
# axes[1].set_title('recovered surface'); axes[1].set_xlabel('K'); axes[1].set_ylabel('T')
# plt.colorbar(im1, ax=axes[1])

# # error heatmap — where does it blow up?
# err = np.abs(sigma_recovered - sigma_true)
# im2 = axes[2].imshow(err, aspect='auto', origin='lower',
#                      extent=[K_grid[0], K_grid[-1], T_grid[0], T_grid[-1]],
#                      vmax=0.05)   # cap colour so structure is visible
# axes[2].set_title('abs error (capped at 0.05)'); axes[2].set_xlabel('K'); axes[2].set_ylabel('T')
# plt.colorbar(im2, ax=axes[2])

# plt.tight_layout(); plt.show()

In [8]:
# j = np.argmin(np.abs(K_grid - 111))
# for T_target in [0.4, 0.8, 1.2, 1.4]:
#     i = np.argmin(np.abs(T_grid - T_target))
#     rec = sigma_recovered[i, j]; tru = sigma_true[0, j]
#     print(f"K=111, T={T_grid[i]:.2f}: recovered {rec:.4f}, true {tru:.4f}, err {abs(rec-tru):.4f}")

In [9]:
# # in the case-b context, after extraction
# dCdK = np.gradient(C, K_grid, axis=1)
# dK = K_grid[1] - K_grid[0]
# d2 = np.empty_like(C)
# d2[:, 1:-1] = (C[:, 2:] - 2*C[:, 1:-1] + C[:, :-2]) / dK**2
# i = np.argmin(np.abs(T_grid - 1.4)); j = np.argmin(np.abs(K_grid - 111))
# print(f"density at K=111,T=1.4: {d2[i,j]:.3e}, "
#       f"row peak: {d2[i].max():.3e}, "
#       f"fraction: {d2[i,j]/d2[i].max():.1%}")

In [10]:
# --- region you actually want trustworthy ---
K_report = (85, 115)
T_report = (0.2, 1.5)

# --- padded grid: extend beyond the report region ---
# pad enough that the report edges sit on INTERIOR (central-difference) nodes
K_grid = np.linspace(75, 125, 41)      # padded strikes (was 85-115)
T_grid = np.linspace(0.1, 1.7, 33)     # padded maturities (was 0.2-1.5)

# price the full padded grid (more solves, but the edges are throwaway)
C = np.zeros((len(T_grid), len(K_grid)))
for j, K in enumerate(K_grid):
    for i, T in enumerate(T_grid):
        C[i, j] = price_call_localvol(K=K, S0=S0, T=T, r=r, q=q,
                                      local_vol_fn=linear_skew,
                                      sigma_max=sigma_max, n_S=200, n_t=200)

sigma_recovered = dupire_local_vol(C, K_grid, T_grid, r, q, verbose=True)

# --- trim to the report region (drop the padding where derivatives were one-sided) ---
K_mask = (K_grid >= K_report[0]) & (K_grid <= K_report[1])
T_mask = (T_grid >= T_report[0]) & (T_grid <= T_report[1])
sigma_report = sigma_recovered[np.ix_(T_mask, K_mask)]
K_report_grid = K_grid[K_mask]
T_report_grid = T_grid[T_mask]

# truth on the report region
sigma_true_report = linear_skew(K_report_grid[np.newaxis, :], None)
err = np.abs(sigma_report - sigma_true_report)
print(f"\nReport region recovery: max {np.nanmax(err):.4f}, mean {np.nanmean(err):.4f}")

NameError: name 'price_call_localvol' is not defined

In [ ]:
# locate the 0.88 max in the REPORT region
err = np.abs(sigma_report - sigma_true_report)
i, j = np.unravel_index(np.nanargmax(err), err.shape)
print(f"worst report point: T={T_report_grid[i]:.3f}, K={K_report_grid[j]:.1f}, "
      f"recovered={sigma_report[i,j]:.4f}, true={sigma_true_report[0,j]:.4f}")

In [ ]:
i = np.argmin(np.abs(T_grid - 1.0))
j = np.argmin(np.abs(K_grid - 102.5))
print("C around the bad point (rows=T, cols=K):")
print(C[i-1:i+2, j-2:j+3])
# compare to BSM-ish expectation: prices should decrease smoothly left-to-right (rising K)
# and be smooth top-to-bottom (rising T)

In [ ]:
from matplotlib import pyplot as plt

plt.figure(figsize=(8,5))
plt.imshow(err, aspect='auto', origin='lower', vmax=0.05,
           extent=[K_report_grid[0], K_report_grid[-1], T_report_grid[0], T_report_grid[-1]])
plt.colorbar(label='abs error (capped 0.05)')
plt.xlabel('K'); plt.ylabel('T'); plt.title('report-region error')
plt.show()

# Lessons: Dupire Implementation & Validation

## What we built

A full Dupire round-trip, validated on synthetic ground truth before any real data:

1. **`calibration/dupire.py`** - extracts $\sigma_{\text{loc}}(K,T)$ from a call-price grid
   via FD derivatives + Dupire's formula, with a density-floor mask for the wings.
2. **`pricing/pde.py`** - a Crank-Nicolson local-vol PDE pricer (log-space grid,
   node-dependent vol) to *generate* prices under a known surface.
3. **The round-trip**: price under known $\sigma_{\text{loc}}^{\text{true}}$ -> extract ->
   check recovery. Pricing and extraction are inverse operations; if both are correct the
   round-trip is the identity (up to numerical error).

## Validation discipline (two stacked checkpoints)

- **Checkpoint 1**: PDE pricer with *constant* vol must reproduce `bsm_price`. Confirmed
  to second order - error quartered on each grid doubling (ratios 4.11, 4.13, 4.63),
  matching CN's $O(\Delta x^2, \Delta t^2)$. The *rate* certifies the scheme, not the
  single-point value.
- **Checkpoint 2 (case b)**: PDE under a known linear skew ($\beta = -0.5$) ->
  `dupire_local_vol` recovers the skew. Mean error 0.008 over a surface spanning
  0.186-0.216, with error concentrated only in the high-$K$/long-$T$ corner.

## PDE pricer design notes

- **Log-space** ($x = \ln S$): turns the variable-coefficient $S^2\partial_{SS}$,
  $S\partial_S$ terms into *constant*-coefficient $\partial_{xx}$, $\partial_x$; a uniform
  $x$-grid is geometric in $S$ (dense where curvature lives).
- **Crank-Nicolson** chosen over explicit (conditionally stable - $\Delta t \lesssim
  \Delta x^2/\sigma_{\max}^2$, too restrictive) and implicit (stable but only 1st-order).
  CN is unconditionally stable AND 2nd-order, one tridiagonal solve per step.
- **Node-dependent vol**: $\sigma_{\text{loc}}$ varies per node and per time level, so the
  operator's diagonals are rebuilt each step - local vol's only footprint on the solver.
- **Boundaries**: deep-OTM $\to 0$, deep-ITM $\to Se^{-q\tau} - Ke^{-r\tau}$; accurate
  because the domain is $\pm 5$ std wide.

## Two bugs caught (both silent, both instructive)

- **Nested `np.gradient` for the second derivative -> sawtooth.** `np.gradient` central-
  differences skip the centre node; nesting them decouples even/odd nodes into independent
  sub-grids that drift apart, giving an alternating zigzag in the recovered surface. **Fix:
  the direct stencil** $(C_{i+1} - 2C_i + C_{i-1})/\Delta K^2$, which uses the centre node
  ($-2C_i$) and couples neighbours. The first derivative via `np.gradient` is fine; only
  the *second* derivative breaks.
- **Too-weak test surface.** $\beta = -0.1$ gave a near-flat surface (0.197-0.203) -
  indistinguishable from case (a), so it tested nothing. Needed $\beta = -0.5$ for a real,
  recoverable slope. A validation surface must actually exercise the thing being tested.

## The denominator curse, confirmed twice

The recovered error lives *exactly* where the density $\partial^2 C/\partial K^2$ thins:
the high-$K$/long-$T$ corner (worst point T=1.4, K=111). This is the structural
instability derived from the formula - density in the denominator -> blow-up where mass is
thin - now seen empirically on clean synthetic data. The hard density floor ($10^{-6}$)
catches only *catastrophic* points; *mild* corner instability passes it. A density-relative
or curvature-aware mask is the real-data refinement (TODO).

## Limitation (motivates Heston, Week 4)

Local vol makes $\sigma$ a *deterministic* function of $(S, t)$ - volatility is fully
determined by where the stock is, with no independent randomness. It fits the smile
exactly, but gets the smile's *dynamics* wrong (the smile rolls the wrong way as spot
moves). Stochastic-vol models (Heston) give volatility its own Brownian motion to fix this.

## Debugging the case-(b) recovery: a four-hypothesis chase

The round-trip initially showed large, structured errors. Four hypotheses, each
*falsified by a targeted measurement* before the right cause emerged:

1. **Denominator/thin density?** Checked the density fraction at the worst point
   (K=111, T=1.4): **88% of row peak** - density was perfectly healthy there. Falsified.
2. **Relative mask too lax (tune $\epsilon$)?** Raising $\epsilon$ would mask healthy
   points to catch this one - and the cause wasn't density anyway. Wrong tool.
3. **Edge-of-grid $\partial_T$ (one-sided `np.gradient`)?** A T-sweep at fixed K showed
   error growing toward the T-boundary - *partially* right, but incomplete.
4. **A single corrupted input price?** Printed the price neighbourhood - immaculate,
   smooth and convex. Falsified.
5. **Actual cause**: the round-trip priced on a **coarse PDE grid (n=100)** for speed,
   whose price error (~$10^{-2}$, per the convergence table) was then **amplified by the
   $\partial_T$ finite difference** - worst at long T where the true $\partial_T$ is
   smallest (theta decays), so noise-to-signal is highest. The error map showed a
   **horizontal T-band** (error grows with T, uniform in K) - the decisive clue.

**Fix**: re-price at n=200 (validated checkpoint-1 resolution, ~$2\times10^{-3}$ error).
Max report error 0.88 -> 0.026, mean 0.054 -> 0.003. Band gone; only a mild residual at
the extreme high-K/long-T corner (genuine edge+density effect, now visible once the
dominant noise was removed).

## Two general principles from this

- **Finite differences amplify the noise floor of their input.** The extraction
  *differentiates* the PDE prices, so it is far more sensitive to price noise than a
  direct price comparison. A grid resolution that is "fine enough" for pricing can be too
  coarse for *differentiating* those prices. (Same root cause as the earlier sawtooth.)
- **Visualize the whole error field early.** The heatmap revealed a T-band structure that
  no single-point probe could - and the *spatial structure of the error is the diagnosis*
  (T-band -> a T-derivative/PDE-noise issue, not a corner or density issue).